# Quality - Validate training_dataset_v0

Valida claves, calidad global y targets futuros de la tabla Gold v0.

In [ ]:
from pyspark.sql import functions as F

LEVEL_TABLE = 'weather.silver.river_levels_daily'
TEMP_TABLE = 'weather.silver.temperature_daily'
RAIN_TABLE = 'weather.silver.rainfall_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
GOLD_TABLE = 'weather.gold.training_dataset_v0'
PUNTO_PREDICCION = 'ana_74100000'
TARGET_STATION = '74100000'

In [ ]:
def assert_table_has_rows(table_name, filter_expr=None):
    df = spark.table(table_name)
    if filter_expr is not None:
        df = df.filter(filter_expr)
    row_count = df.count()
    print(f'{table_name}: {row_count} rows')
    if row_count == 0:
        raise ValueError(f'{table_name} has no rows')


def assert_unique(table_name, key_cols, filter_expr=None):
    df = spark.table(table_name)
    if filter_expr is not None:
        df = df.filter(filter_expr)
    duplicates = df.groupBy(*key_cols).count().filter(F.col('count') > 1)
    duplicate_count = duplicates.count()
    print(f'{table_name} duplicate keys on {key_cols}: {duplicate_count}')
    if duplicate_count > 0:
        duplicates.show(20, truncate=False)
        raise ValueError(f'{table_name} has duplicate keys')


def latest_quality(source_table, attribute_name):
    rows = (
        spark.table(QUALITY_TABLE)
        .filter(F.col('source_table') == F.lit(source_table))
        .filter(F.col('attribute_name') == F.lit(attribute_name))
        .filter(F.col('grain') == F.lit('global_source_daily'))
        .orderBy(F.col('evaluated_at').desc_nulls_last())
        .limit(1)
        .collect()
    )
    if not rows:
        raise ValueError(f'Missing attribute_quality for {source_table}.{attribute_name}')
    return rows[0]


# Valida contra caudal_actual_m3s: desde la Decision 040 el nivel del target no esta en
# Gold, y el caudal es el unico target. Antes esto chequeaba nivel_rio_t_mas_* contra
# nivel_rio_actual_m, o sea el target secundario en vez del principal.
def assert_future_target(horizon_days, target_col):
    base = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).alias('base')
    future = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).alias('future')
    mismatches = (
        base.join(
            future,
            F.date_add(F.col('base.fecha'), horizon_days) == F.col('future.fecha'),
            'inner',
        )
        .filter(F.col(f'base.{target_col}').isNotNull())
        .filter(F.col('future.caudal_actual_m3s').isNotNull())
        .filter(F.abs(F.col(f'base.{target_col}') - F.col('future.caudal_actual_m3s')) > F.lit(0.000001))
    )
    mismatch_count = mismatches.count()
    print(f'{target_col} mismatches: {mismatch_count}')
    if mismatch_count > 0:
        mismatches.select('base.fecha', F.col(f'base.{target_col}'), F.col('future.fecha'), F.col('future.caudal_actual_m3s')).show(20, truncate=False)
        raise ValueError(f'{target_col} does not match future discharge')

In [ ]:
for table_name in [LEVEL_TABLE, TEMP_TABLE, QUALITY_TABLE, GOLD_TABLE]:
    spark.sql(f'DESCRIBE {table_name}').show(truncate=False)

assert_table_has_rows(LEVEL_TABLE, F.col('codigoestacao') == F.lit(TARGET_STATION))
assert_table_has_rows(TEMP_TABLE)
assert_table_has_rows(QUALITY_TABLE)
assert_table_has_rows(GOLD_TABLE, F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

assert_unique(LEVEL_TABLE, ['fecha', 'codigoestacao'], F.col('codigoestacao') == F.lit(TARGET_STATION))
assert_unique(TEMP_TABLE, ['fecha', 'estacion_id'])
assert_unique(GOLD_TABLE, ['fecha', 'punto_prediccion'], F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

for source_table, attribute_name in [
    (LEVEL_TABLE, 'nivel_media_cm'),
    (TEMP_TABLE, 'temp_media_c'),
    (TEMP_TABLE, 'temp_min_c'),
    (TEMP_TABLE, 'temp_max_c'),
    (RAIN_TABLE, 'lluvia_acumulada_mm'),
]:
    quality = latest_quality(source_table, attribute_name)
    print(f"quality {source_table}.{attribute_name}: missing_pct={quality['missing_pct']}, is_usable={quality['is_usable']}")

gold = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

# R8 (Decision 019): no hay porton binario para lluvia. Se valida que la cobertura
# publicada sea coherente, en vez de un missing_pct global que ya no bloquea nada.
bad_coverage_rows = gold.filter(
    (F.col('lluvia_agregado_alta_frontera_cobertura_pct') < 0) | (F.col('lluvia_agregado_alta_frontera_cobertura_pct') > 1)
).count()
print(f'rows with lluvia_agregado_alta_frontera_cobertura_pct out of [0,1]: {bad_coverage_rows}')
if bad_coverage_rows > 0:
    raise ValueError('lluvia_agregado_alta_frontera_cobertura_pct out of the [0,1] range')

stale_gate_rows = gold.filter(F.col('lluvia_is_usable').isNotNull()).count()
print(f'rows with lluvia_is_usable populated (deprecated column, should stay NULL): {stale_gate_rows}')
if stale_gate_rows > 0:
    raise ValueError('lluvia_is_usable should be NULL under R8; the old all-or-nothing gate is deprecated')

# R8 aplicado a temperatura (Fase 3, Decision 025): mismo assert de cobertura que lluvia.
bad_temp_coverage_rows = gold.filter(
    (F.col('temp_agregado_alta_frontera_cobertura_pct') < 0) | (F.col('temp_agregado_alta_frontera_cobertura_pct') > 1)
).count()
print(f'rows with temp_agregado_alta_frontera_cobertura_pct out of [0,1]: {bad_temp_coverage_rows}')
if bad_temp_coverage_rows > 0:
    raise ValueError('temp_agregado_alta_frontera_cobertura_pct out of the [0,1] range')

# Observacion en grilla CPTEC (Decision 033): misma validacion de cobertura que lluvia/temperatura
# por estacion, mas un chequeo de rango fisico (la fuente es externa y se regenera; un cambio de
# unidades o de mascara apareceria aca antes que en un modelo).
for coverage_col in ['lluvia_merge_alta_frontera_cobertura_pct', 'temp_samet_alta_frontera_cobertura_pct']:
    bad = gold.filter((F.col(coverage_col) < 0) | (F.col(coverage_col) > 1)).count()
    print(f'rows with {coverage_col} out of [0,1]: {bad}')
    if bad > 0:
        raise ValueError(f'{coverage_col} out of the [0,1] range')

bad_merge_range = gold.filter((F.col('lluvia_merge_alta_frontera_mm') < 0) | (F.col('lluvia_merge_alta_frontera_mm') > 400)).count()
bad_samet_range = gold.filter((F.col('temp_samet_alta_frontera_media_c') < -15) | (F.col('temp_samet_alta_frontera_media_c') > 45)).count()
print(f'rows with lluvia_merge_alta_frontera_mm outside [0,400]: {bad_merge_range}; temp_samet_alta_frontera_media_c outside [-15,45]: {bad_samet_range}')
if bad_merge_range > 0 or bad_samet_range > 0:
    raise ValueError('MERGE/SAMeT values out of physical range')

cptec_cov = gold.agg(
    F.avg(F.when(F.col('lluvia_merge_alta_frontera_mm').isNotNull(), 1.0).otherwise(0.0)).alias('merge_non_null_pct'),
    F.avg(F.when(F.col('temp_samet_alta_frontera_media_c').isNotNull(), 1.0).otherwise(0.0)).alias('samet_non_null_pct'),
).first()
print(f"MERGE non-null pct in Gold: {cptec_cov['merge_non_null_pct']}; SAMeT non-null pct: {cptec_cov['samet_non_null_pct']}")

# Los 8 horizontes de la Decision 019, todos sobre el unico target (caudal).
for horizon_days in (1, 2, 3, 4, 5, 6, 7, 14):
    assert_future_target(horizon_days, f'caudal_t_mas_{horizon_days}d')

gold.agg(
    F.min('fecha').alias('inicio'),
    F.max('fecha').alias('fin'),
    F.count('*').alias('rows'),
    F.countDistinct('punto_prediccion').alias('puntos'),
    F.sum(F.when(F.col('caudal_actual_m3s').isNull(), 1).otherwise(0)).alias('caudal_null_rows'),
    F.sum(F.when(F.col('caudal_t_mas_1d').isNull(), 1).otherwise(0)).alias('target_1d_null_rows'),
).show(truncate=False)
# ---------------------------------------------------------------------------
# Guarda de fuga (Decision 043). Ninguna feature puede predecir el target mejor
# que el caudal de hoy.
#
# El umbral tiene sentido fisico, no es arbitrario: la autocorrelacion del rio es el
# mejor predictor legitimo que existe -- nada deberia anticipar el caudal de manana
# mejor que el caudal de hoy. Una feature que lo supere esta mirando adelante.
#
# Por que hace falta como assert permanente y no como revision de una vez: la fuga que
# motivo la Decision 040 (nivel_rio_t_mas_*, que es el target en otras unidades via curva
# de aforo) no se detectaba por nombre ni por tipo. Comparar valores encuentra columnas
# duplicadas; un proxy del futuro no es identico a nada y pasa igual. Esta prueba lo
# agarra porque mide capacidad predictiva, no parecido.
MARGEN_CORR = 0.005

identificadores = {'fecha', 'punto_prediccion', 'codigoestacao', 'feature_generated_at', 'updated_at'}
numericas = {
    f.name for f in spark.table(GOLD_TABLE).schema.fields
    if f.dataType.typeName() in ('double', 'float', 'integer', 'long', 'decimal')
}
features = sorted(numericas - identificadores - {c for c in numericas if '_t_mas_' in c})

base_corr = abs(gold.stat.corr('caudal_actual_m3s', 'caudal_t_mas_1d'))
print(f'linea base |corr(caudal_actual_m3s, caudal_t_mas_1d)| = {base_corr:.4f}')

sospechosas = []
for columna in features:
    try:
        r = gold.stat.corr(columna, 'caudal_t_mas_1d')
    except Exception:
        continue          # columna constante o sin pares validos suficientes
    if r is None:
        continue
    if abs(r) > base_corr + MARGEN_CORR:
        sospechosas.append((columna, abs(r)))

if sospechosas:
    for columna, r in sorted(sospechosas, key=lambda x: -x[1]):
        print(f'  FUGA POSIBLE: {columna} |r|={r:.4f} supera la base {base_corr:.4f}')
    raise ValueError(
        'Hay features que predicen el target mejor que el caudal de hoy. O miran adelante '
        '(fuga: revisar como se construyen en ETL_Gold_Training_Dataset_v0), o son un pronostico '
        'legitimo emitido en t -- las features de la Fase 4 pueden caer aca de forma valida y '
        'en ese caso corresponde documentarlas y excluirlas explicitamente de esta guarda.'
    )
print(f'Sin fuga: ninguna de las {len(features)} features supera la autocorrelacion del rio.')
